# Notebook 3: Vector DB Indexing

**Goal**: Build FAISS and ChromaDB indexes, compare index types, and benchmark search performance.

## Topics:
- FAISS Flat vs HNSW index comparison
- ChromaDB with metadata filtering
- Index persistence and reload
- Search latency benchmarks


In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from data_pipeline.document_loader import DocumentLoader
from data_pipeline.chunking import ChunkingEngine
from embeddings.embedding import EmbeddingEngine
from vector_store.vector_store import FAISSVectorStore, ChromaVectorStore, VectorStoreFactory

print('Imports OK')

## 3.1 Prepare Chunks and Embeddings

In [ ]:
# Load and chunk documents
loader = DocumentLoader()
docs = loader.load('../sample_data/', loader_type='directory')

chunker = ChunkingEngine(strategy='recursive', chunk_size=200, chunk_overlap=30)
chunks = chunker.split(docs)

# Embed
engine = EmbeddingEngine(
    provider='sentence_transformers',
    model_name='all-MiniLM-L6-v2',
    use_cache=True,
    cache_dir='../.embed_cache',
)

texts = [c.content for c in chunks]
result = engine.encode(texts)
vectors = result.embeddings
ids = [c.chunk_id for c in chunks]
metadatas = [c.metadata for c in chunks]

print(f'Ready to index: {len(chunks)} chunks, shape={vectors.shape}')

## 3.2 FAISS Flat Index (Exact Search)

In [ ]:
faiss_flat = FAISSVectorStore(
    dimension=engine.dimension,
    index_type='flat',
    metric='cosine',
)

t0 = time.perf_counter()
faiss_flat.add(ids, vectors, texts, metadatas)
add_ms = (time.perf_counter() - t0) * 1000

print(f'FAISS Flat Index:')
print(f'  Vectors indexed: {faiss_flat.count}')
print(f'  Index time:      {add_ms:.1f} ms')
print(f'  Index type:      Exact (Flat)')

## 3.3 FAISS HNSW Index (ANN Search)

In [ ]:
faiss_hnsw = FAISSVectorStore(
    dimension=engine.dimension,
    index_type='hnsw',
    metric='cosine',
    hnsw_m=16,
    hnsw_ef_construction=200,
    hnsw_ef_search=50,
)

t0 = time.perf_counter()
faiss_hnsw.add(ids, vectors, texts, metadatas)
hnsw_add_ms = (time.perf_counter() - t0) * 1000

print(f'FAISS HNSW Index:')
print(f'  Vectors indexed: {faiss_hnsw.count}')
print(f'  Index time:      {hnsw_add_ms:.1f} ms')
print(f'  M parameter:     16 (connections per layer)')
print(f'  ef_construction: 200 (build-time quality)')
print(f'  ef_search:       50 (query-time quality)')

## 3.4 Search Benchmark

In [ ]:
queries = [
    'What is the refund policy?',
    'How to track my order?',
    'Laptop price and specifications',
    'International shipping',
    'Payment options',
]

n_trials = 20
flat_times, hnsw_times = [], []

for q in queries * (n_trials // len(queries)):
    qvec = engine.encode_query(q)
    
    t0 = time.perf_counter()
    flat_results = faiss_flat.search(qvec, top_k=5)
    flat_times.append((time.perf_counter() - t0) * 1000)
    
    t0 = time.perf_counter()
    hnsw_results = faiss_hnsw.search(qvec, top_k=5)
    hnsw_times.append((time.perf_counter() - t0) * 1000)

print(f'Search Benchmark (n={n_trials} queries, top-5):')
print(f'  Flat  — p50: {np.percentile(flat_times,50):.2f}ms  p90: {np.percentile(flat_times,90):.2f}ms')
print(f'  HNSW  — p50: {np.percentile(hnsw_times,50):.2f}ms  p90: {np.percentile(hnsw_times,90):.2f}ms')
print()
print('Note: For this small dataset, Flat may be faster.')
print('HNSW advantage becomes significant at 100k+ vectors.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(flat_times, bins=15, alpha=0.6, label='Flat (exact)', color='blue')
ax.hist(hnsw_times, bins=15, alpha=0.6, label='HNSW (approx)', color='orange')
ax.set_xlabel('Query Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Search Latency Distribution: Flat vs HNSW')
ax.legend()
ax.axvline(np.percentile(flat_times, 50), color='blue', linestyle='--', alpha=0.7, label='Flat p50')
ax.axvline(np.percentile(hnsw_times, 50), color='orange', linestyle='--', alpha=0.7, label='HNSW p50')
plt.tight_layout()
plt.savefig('search_latency.png', dpi=100)
plt.show()

## 3.5 Search with Metadata Filtering

In [ ]:
# Filter results by source file
query = 'What payment methods are accepted?'
qvec = engine.encode_query(query)

# Unfiltered search
all_results = faiss_flat.search(qvec, top_k=5)

# Filtered to a specific source (metadata filtering)
# FAISS does client-side filtering
faq_filter = {'file_type': 'md'}    # only Markdown docs
filtered_results = faiss_flat.search(qvec, top_k=5, filter=faq_filter)

print(f'Query: "{query}"')
print(f'\n--- Unfiltered (top 3) ---')
for r in all_results[:3]:
    print(f'  [{r.score:.4f}] ({r.metadata.get("file_type","?")}) {r.content[:80]}...')

print(f'\n--- Filtered to Markdown only ---')
for r in filtered_results[:3]:
    print(f'  [{r.score:.4f}] ({r.metadata.get("file_type","?")}) {r.content[:80]}...')

## 3.6 ChromaDB (Embedded Mode)

In [ ]:
chroma = ChromaVectorStore(
    collection_name='demo_collection',
    persist_dir='../chroma_data',
)

# Clean metadata for Chroma (values must be str/int/float/bool)
def clean_meta(m):
    return {k: str(v) if not isinstance(v, (str, int, float, bool)) else v 
            for k, v in m.items()}

chroma_metas = [clean_meta(m) for m in metadatas]
chroma.add(ids, vectors, texts, chroma_metas)

print(f'ChromaDB collection count: {chroma.count}')

# Search with Chroma native metadata filter
query_vec = engine.encode_query('shipping policy')
chroma_results = chroma.search(query_vec, top_k=3)

print(f'\nChroma search results for "shipping policy":')
for r in chroma_results:
    print(f'  [{r.score:.4f}] {r.content[:100]}...')

## 3.7 Persist and Reload FAISS Index

In [ ]:
import os

# Persist
save_path = '../saved_index_demo'
faiss_flat.persist(save_path)

# Reload from disk
faiss_reloaded = FAISSVectorStore(dimension=engine.dimension, index_type='flat')
faiss_reloaded.load(save_path)

print(f'Original count:  {faiss_flat.count}')
print(f'Reloaded count:  {faiss_reloaded.count}')

# Verify same results
qvec = engine.encode_query('refund eligibility')
orig_results = faiss_flat.search(qvec, top_k=3)
load_results = faiss_reloaded.search(qvec, top_k=3)

scores_match = all(
    abs(a.score - b.score) < 1e-5 
    for a, b in zip(orig_results, load_results)
)
print(f'Results match after reload: {scores_match} ✓')

## Index Type Decision Guide

| Index Type | Best For | Dataset Size | Accuracy | Memory |
|---|---|---|---|---|
| Flat (exact) | Dev/testing, small datasets | < 100K vectors | 100% exact | n × d × 4 bytes |
| IVF | Balanced speed/accuracy | 100K – 10M | ~98% | Flat + centroids |
| HNSW | Low-latency production | Any (preferred > 10K) | 95-99% | ~2× Flat |
| HNSW + PQ | Memory-constrained | > 10M | 90-95% | 4-8× compression |

**Production recommendation**: Use HNSW (M=32, ef=200) for most cases. Add Product Quantization (PQ) if memory > 32GB becomes a constraint.